# OneDrive Client Logic Verification (Live API)

This notebook demonstrates how the `OneDriveClient` interacts with Microsoft Graph API and how the zero-copy file ingestion to GCS works using real Microsoft Entra ID authentication.

In [ ]:
import sys
from dotenv import load_dotenv
from pydantic import SecretStr
# Append repository root to sys.path
sys.path.append("../..")

load_dotenv()
from mcp_servers.onedrive.app.config import ONEDRIVE_AUTH_CONFIG, ONEDRIVE_SERVER_CONFIG
from mcp_servers.onedrive.app.onedrive_client import OneDriveClient
from mcp_servers.onedrive.app.schemas import *
from azure.identity import InteractiveBrowserCredential

## 1. Authentication (Interactive Browser Flow)
We use `azure.identity.InteractiveBrowserCredential` to authenticate. This will open a browser window for you to log in securely.

In [ ]:
client_id = ONEDRIVE_AUTH_CONFIG.CLIENT_ID
tenant_id = ONEDRIVE_AUTH_CONFIG.TENANT_ID

In [ ]:
if not client_id or not tenant_id:
    raise ValueError("Please configure MICROSOFT_CLIENT_ID and MICROSOFT_TENANT_ID to authenticate.")

print("Initiating Interactive Browser Flow...")
credential = InteractiveBrowserCredential(
    client_id=client_id,
    tenant_id=tenant_id,
    redirect_uri="http://localhost:8000"
)

# azure-identity requires scopes to be fully qualified with the resource URL for Graph API
scopes = [f"https://graph.microsoft.com/{scope}" for scope in ONEDRIVE_AUTH_CONFIG.SCOPES if scope != "offline_access"]

token = credential.get_token(*scopes)

token_value = SecretStr(token.token)

client = OneDriveClient(token_value)
print("\nSuccessfully authenticated with Microsoft Graph API!")

## 2. Finding Files
Let's perform a fuzzy search across the entire OneDrive for a specific item using `find_items`.

In [ ]:
search_req = FindItemsRequest(
    main_folder=MainFolder.SHARED_WITH_ME,
    item_name="O S I R I S",
    page=1,
    # sort_by="creation_date",
    # sort_order="desc"
)
search_results = await client.find_items(search_req)
search_results

In [ ]:
search_req = FindItemsRequest(
    main_folder=MainFolder.MY_FILES,
    item_name="personal data",
    page=1,
)
search_results = await client.find_items(search_req)
search_results

## 3. Listing Folder Contents
Now let's use `list_folder_contents` to explicitly list the contents of a specific folder without recursive fuzzy matching.

In [ ]:
# folder_id = search_results.objects_found[0].folder_id
list_req = ListFolderContentsRequest(
    folder_id="moc-folder-id",
    # page=2
)
folder_contents = await client.list_folder_contents(list_req)
folder_contents

## 4. Reading a File
Finally, let's use `read_file` to ingest a file. This method will fetch the file and stream it into the landing zone in GCS.

In [ ]:

read_req = ReadFileRequest(
    file_id="mock-file-id",
    # use_cache=False
    )

client.read_file(read_req)